# Préparation des données Filosofi

Ce notebook prépare les données socio-économiques communales issues de Filosofi.

**Objectif :**
- récupérer les principaux indicateurs de niveau de vie et de pauvreté ;
- conserver les variables utiles à l’analyse des inégalités territoriales ;
- préparer une table propre au niveau communal.

**Source :** INSEE – Filosofi 2021  
**Clé géographique :** `CODGEO`  
**Sortie :** `filosofi_communes_2021.csv`

In [4]:
import pandas as pd
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [5]:
path_filosofi = "https://minio.lab.sspcloud.fr/mangane/Projet_PDS_2026/raw/base-cc-filosofi-2021-geo2024.xlsx"

filosofi_file = pd.ExcelFile(path_filosofi)

print(filosofi_file.sheet_names)

['COM', 'EPCI', 'EPT', 'ARR', 'DEP', 'CTCD', 'REG', 'METRO', 'AAV2020', 'UU2020', 'ZE2020', 'BV2022', 'Variables', 'Documentation générale', 'Documentation revenu disponible']


In [4]:
variables_filosofi = pd.read_excel(
    path_filosofi,
    sheet_name="Variables"
)

pd.set_option("display.max_columns", None)

print(variables_filosofi.head(40).to_string())

                      Fichier Localisé Social et Fiscal (FiLoSoFi) - Année 2021                                             Unnamed: 1                                                                                     Unnamed: 2
0                                                                     Variables                                                    NaN                                                                                            NaN
1                    Mise en ligne le 19/07/2024       Géographie au 01/01/2024                                                    NaN                                                                                            NaN
2   Source(s) : Insee-DGFIP-Cnaf-Cnav-CCMSA, Fichier localisé social et fiscal.                                                    NaN                                                                                            NaN
3                                                                           NaN 

In [8]:
df_filosofi = pd.read_excel(
    path_filosofi,
    sheet_name="COM",
    header=5,
    dtype={"CODGEO": str},
    na_values=["s"],
    decimal = ","
)

print(df_filosofi.head())
print(df_filosofi.shape)

  CODGEO                   LIBGEO NBMENFISC21 NBPERSMENFISC21  MED21 PIMP21  \
0  01001  L'Abergement-Clémenciat         346             895  25820    NaN   
1  01002    L'Abergement-de-Varey         115             266  24480    NaN   
2  01004        Ambérieu-en-Bugey        6855           15092  21660   50.0   
3  01005      Ambérieux-en-Dombes         800            2028  24610   61.0   
4  01006                  Ambléon          51             107  24210    NaN   

  TP6021 TP60AGE121 TP60AGE221 TP60AGE321 TP60AGE421 TP60AGE521 TP60AGE621  \
0    NaN        NaN        NaN        NaN        NaN        NaN        NaN   
1    NaN        NaN        NaN        NaN        NaN        NaN        NaN   
2   17.0       17.0       22.0       19.0       16.0       11.0        NaN   
3    NaN        NaN        NaN        NaN        NaN        NaN        NaN   
4    NaN        NaN        NaN        NaN        NaN        NaN        NaN   

  TP60TOL121 TP60TOL221 PACT21 PTSA21 PCHO21 PBEN21 PPEN

MED21 = niveau de vie médian 
TP6021 = taux de pauvreté 
RD21 = rapport interdécile, une mesure des inégalités de niveau de vie

In [9]:
df_filosofi = df_filosofi[
    [
        "CODGEO",
        "LIBGEO",
        "MED21",
        "TP6021",
        "RD21"
    ]
].copy()

print(df_filosofi.head())

  CODGEO                   LIBGEO  MED21 TP6021 RD21
0  01001  L'Abergement-Clémenciat  25820    NaN  NaN
1  01002    L'Abergement-de-Varey  24480    NaN  NaN
2  01004        Ambérieu-en-Bugey  21660   17.0  3.1
3  01005      Ambérieux-en-Dombes  24610    NaN  2.6
4  01006                  Ambléon  24210    NaN  NaN


In [10]:
print("Codes communes manquants :", df_filosofi["CODGEO"].isna().sum())
print("Doublons CODGEO :", df_filosofi["CODGEO"].duplicated().sum())

print("\nValeurs manquantes :")
print(df_filosofi.isna().sum())

Codes communes manquants : 0
Doublons CODGEO : 0

Valeurs manquantes :
CODGEO        0
LIBGEO        0
MED21      3593
TP6021    30500
RD21      29549
dtype: int64


In [11]:
print(df_filosofi.isna().sum())

CODGEO        0
LIBGEO        0
MED21      3593
TP6021    30500
RD21      29549
dtype: int64


In [12]:
path_filosofi_clean = (
    "s3://mangane/Projet_PDS_2026/processed/filosofi_communes_2021.csv"
)

df_filosofi.to_csv(
    path_filosofi_clean,
    index=False
)